# 07 - Governed RAG And Embedding Ingestion Gate

Before data enters a RAG index or an embedding store, ask Metatate. Training on
ticket text is a typed deny; LLM inference over customer data is permitted —
the gate keeps the corpus honest either way.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
if mode == "live" and not os.getenv("METATATE_MCP_URL"):
    print("Live mode needs a Metatate endpoint. Fastest path (about 5 minutes):")
    print("  1. Create a free account: https://app.getmetatate.com/sign-up?ref=examples")
    print("  2. Workspace dashboard: 'Load the demo' banner -> 'Load the AcmeCloud demo'")
    print("  3. MCP Tools -> Tokens: issue a token; Connect tab has your endpoint URL")
    print("  4. export METATATE_MCP_URL=... METATATE_SAAS_MCP_TOKEN=...")
    print("     (full steps: docs/live-mode-saas.md)")

client = get_client()
print(f"Metatate examples mode: {mode}")


def asset(table, column=None):
    ref = {"database": "acmecloud_demo", "schema": "public", "table": table}
    if column:
        ref["column"] = column
    return ref


def answer_label(answer):
    state = answer.get("state")
    if state and state != "answered":
        return state
    return answer.get("decision") or answer.get("verdict") or "unknown"


def print_answer(answer):
    print(f"state:    {answer.get('state')}")
    if "decision" in answer:
        print(f"decision: {answer['decision']}")
    if "verdict" in answer:
        print(f"verdict:  {answer['verdict']}")
    if answer.get("reason"):
        print(f"reason:   {answer['reason']}")
    for condition in answer.get("conditions") or []:
        print(f"condition [{condition.get('kind')}]: {condition.get('requirement')}")
    for prohibition in answer.get("prohibitions") or []:
        print(f"prohibition: {prohibition.get('detail')}")
    for obligation in answer.get("obligations") or []:
        print(f"obligation [{obligation.get('type')}]: {obligation.get('target')}")
    if "can_proceed_now" in answer:
        print(f"can_proceed_now: {answer['can_proceed_now']}")


In [ ]:
candidates = [
    {
        "corpus": "support ticket bodies (fine-tune)",
        "answer": client.authorize_use(
            asset("support_tickets"),
            use="fine-tune a support assistant on ticket text",
            scenario_key="ai.training",
        ),
    },
    {
        "corpus": "customer account summaries (LLM inference)",
        "answer": client.authorize_use(
            asset("customers"),
            use="summarize customer accounts with an LLM",
            scenario_key="ai.inference",
        ),
    },
]
for candidate in candidates:
    answer = candidate["answer"]
    label = answer_label(answer)
    action = "INGEST" if label == "allow" else "SKIP"
    print(f"{action} {candidate['corpus']} -> {label}")
    if answer.get("reason"):
        print(f"  {answer['reason']}")


## Validate the retrieval query that will feed the index


In [ ]:
retrieval_sql = client.validate_query_context(
    "SELECT region, SUM(arr) FROM customers GROUP BY region",
    scenario_key="purpose.allowed_use",
    default_database="acmecloud_demo",
    default_schema="public",
)
print(f"retrieval query verdict: {retrieval_sql['verdict']}")
